# NLP Preprocessing


We will use a short excerpt from a corporate earnings call transcript (Microsoft Q2 2026) to demonstrate common NLP preprocessing steps.


In [ ]:
# Excerpt from Satya Nadella's Opening Remarks
text = """
You can think of agents as the new apps, and to build, deploy, and manage agents, 
customers will need a model catalog, tuning services, harness for orchestration, 
services for context engineering, AI safety, management, observability, and security. 
It starts with having broad model choice. Our customers expect to use multiple models 
as part of any workload that they can fine-tune and optimize based on cost, latency, 
and performance requirements. We offer the broadest selection of models of any 
hyperscaler. This quarter, we added support for GPT-5 too, as well as Claude 4.5. 
Already, over 1,500 customers have used both Anthropic and OpenAI models on Foundry. 
We are seeing increasing demand for region-specific models, including Mistral and 
Cohere, as more customers look for sovereign AI choices.
"""

:::{note} Why triple double quotes?

Using triple double quotes (`"""`) allows us to include multi-line text without needing to escape newlines or other special characters. This is especially useful for large blocks of text, such as transcripts, where we want to preserve the formatting and structure of the original content. You can also use triple single quotes (`'''`) for the same purpose.

:::


First, install spaCy and download the English language model:

If using pip:

```bash
pip install spacy
```

Then download the language model:

```bash
python -m spacy download en_core_web_sm
```

If using conda:

```bash
conda install -c conda-forge spacy
```

Then download the language model:

```bash
python -m spacy download en_core_web_sm
```


:::{note} What is `en_core_web_sm`?

`en_core_web_sm` is a small English language model provided by spaCy. It includes vocabulary, syntax, and entities, and is suitable for many NLP tasks. For more advanced features, larger models like `en_core_web_md` or `en_core_web_lg` can be used.

There are other language models available for different languages and sizes. You can find more information in the [spaCy documentation](https://spacy.io/models).

:::


## Common NLP Preprocessing Steps

Recall from the previous page that raw text often needs to be cleaned before analysis.

Common preprocessing steps include:

- Lowercasing
- Removing punctuation
- Tokenization
- Stopword removal
- Stemming or lemmatizatio

[spaCy](https://spacy.io/) is a powerful NLP library that provides built-in functions for many of these preprocessing steps. We will use it to demonstrate how to preprocess our conference call transcript excerpt.


Load the model.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

### Parse the Text as a spaCy Document

The first step is to parse the raw text into a spaCy `Doc` object.


In [ ]:
doc = nlp(text)

### Sentence Segmentation

spaCy can automatically split text into sentences.


In [ ]:
for sent in doc.sents:
    print("---------")
    print(sent.text.strip())

---------
You can think of agents as the new apps, and to build, deploy, and manage agents, 
customers will need a model catalog, tuning services, harness for orchestration, 
services for context engineering, AI safety, management, observability, and security.
---------
It starts with having broad model choice.
---------
Our customers expect to use multiple models 
as part of any workload that they can fine-tune and optimize based on cost, latency, 
and performance requirements.
---------
We offer the broadest selection of models of any 
hyperscaler.
---------
This quarter, we added support for GPT-5 too, as well as Claude 4.5.
---------
Already, over 1,500 customers have used both Anthropic and OpenAI models on Foundry.
---------
We are seeing increasing demand for region-specific models, including Mistral and 
Cohere, as more customers look for sovereign AI choices.


### Tokenization

Tokenization splits text into individual units called tokens.


In [ ]:
for token in doc[:10]:
    print(token.text)



You
can
think
of
agents
as
the
new
apps


### Token Attributes

spaCy provides built-in metadata about each token.


In [ ]:
for token in doc[:10]:
    print(token.text, token.pos_, token.is_stop)


 SPACE False
You PRON True
can AUX True
think VERB False
of ADP True
agents NOUN False
as ADP True
the DET True
new ADJ False
apps NOUN False


We can format this into a Pandas DataFrame.


In [ ]:
import pandas as pd

data = [
    {"token": token.text, "pos": token.pos_, "is_stop": token.is_stop}
    for token in doc[:10]
]

df = pd.DataFrame(data)
df

,token,pos,is_stop
0,\n,SPACE,False
1,You,PRON,True
2,can,AUX,True
3,think,VERB,False
4,of,ADP,True
5,agents,NOUN,False
6,as,ADP,True
7,the,DET,True
8,new,ADJ,False
9,apps,NOUN,False


Useful attributes include:

| Attribute        | Description                    |
| ---------------- | ------------------------------ |
| `token.pos_`     | Part-of-speech                 |
| `token.lemma_`   | Base form of the word          |
| `token.is_stop`  | Whether the word is a stopword |
| `token.is_punct` | Whether it is punctuation      |
| `token.like_num` | Whether it represents a number |


### Lowercasing

spaCy tokens contain both original and normalized text.


In [ ]:
tokens_lower = [token.text.lower() for token in doc if not token.is_punct]

tokens_lower[:10]

['\n', 'you', 'can', 'think', 'of', 'agents', 'as', 'the', 'new', 'apps']

### Removing Stopwords and Puntuation


In [ ]:
clean_tokens = [
    token.text.lower() for token in doc if not token.is_stop and not token.is_punct
]

clean_tokens[:10]

['\n',
 'think',
 'agents',
 'new',
 'apps',
 'build',
 'deploy',
 'manage',
 'agents',
 '\n']

### Lemmatization

Lemmatization reduces words to their base form.


In [ ]:
lemmas = [
    token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct
]

lemmas[:10]

['\n',
 'think',
 'agent',
 'new',
 'app',
 'build',
 'deploy',
 'manage',
 'agent',
 '\n']

## Feature Extraction

spaCy can also extract more complex features like named entities, noun chunks, and syntactic dependencies.


### Part-of-Speech Tagging

spaCy can identify grammatical roles.


In [ ]:
for token in doc[:10]:
    print(token.text, token.pos_)


 SPACE
You PRON
can AUX
think VERB
of ADP
agents NOUN
as ADP
the DET
new ADJ
apps NOUN


### Named Entity Recognition

Named Entity Recognition (NER) identifies important entities in text.


In [ ]:
for ent in doc.ents:
    print(ent.text, ent.label_)

AI GPE
This quarter DATE
Claude 4.5 LAW
over 1,500 CARDINAL
OpenAI ORG
Foundry ORG
Mistral ORG


Common entity types:

| Label  | Meaning        |
| ------ | -------------- |
| PERSON | Person name    |
| ORG    | Organization   |
| GPE    | Location       |
| DATE   | Date           |
| MONEY  | Monetary value |


In the next page, we will see how to use these features for analysis.
